![JohnSnowLabs](https://sparknlp.org/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp/blob/master/examples/python/transformers/onnx/HuggingFace_ONNX_in_Spark_NLP_LanguageDetectorDL.ipynb)

## Import fastText language identification models (GlotLID) from HuggingFace 🤗 into Spark NLP 🚀

- `LanguageDetectorDL` runs supervised fastText language identification models on ONNX since `Spark NLP 7.0.1`.
- This example imports [GlotLID v4](https://huggingface.co/cis-lmu/glotlid) (`model_v4.bin`): 2,060 labels written as ISO 639-3 code plus ISO 15924 script, such as `eng_Latn` or `cmn_Hani`, and non-language labels for code (`zxx_Code`), maths (`zxx_Zmth`) and symbols (`zxx_Zsym`). Its licence is Apache 2.0, with additional notices about the training data in the model's LICENSE file.
- Only fastText models trained with softmax loss and `wordNgrams=1` can be exported this way. GlotLID v4 qualifies; GlotLID v4 small (hierarchical softmax) and the quantized `.ftz` files do not.
- The exported model is 782 MB. The last section makes a 196 MB int8 version.

## Export the model to ONNX

- `fasttext-turbo` reads the fastText model (GlotLID recommends it; the original `fasttext` package works too), `onnx` writes the graph and `onnxruntime` makes the optional int8 version. Spark NLP itself needs none of them.

In [ ]:
!pip install -q fasttext-turbo onnx onnxruntime huggingface_hub

- We pin the file to `model_v4.bin`; `model.bin` on the hub always points to the latest GlotLID version.

In [ ]:
from huggingface_hub import hf_hub_download

MODEL_PATH = hf_hub_download(repo_id="cis-lmu/glotlid", filename="model_v4.bin")
EXPORT_DIR = "glotlid_v4_onnx"

A fastText prediction has two parts:

1. **Text to ids.** The text is split on whitespace; each known word adds its id, and every word adds ids for its 2- to 5-character pieces (hashed into 2 million buckets). Spark NLP does this part in Scala, exactly as fastText does, so it is not in the graph.
2. **Ids to scores.** The rows of the embedding table for those ids are averaged, multiplied by the output layer and turned into probabilities with a softmax. This part becomes the ONNX graph.

Spark NLP expects this folder layout:

- `model.onnx`: takes `input_ids` and `mask` (both `[batch, features]`) and returns the softmax `probs` (`[batch, labels]`)
- `assets/language.txt`: the labels, in output order
- `assets/vocab.json`: the words, in id order
- `assets/config.json`: `minn`, `maxn`, `bucket` and `nwords`, the settings of the text-to-ids part

In [ ]:
import json
import os

import fasttext
import numpy as np
import onnx
from onnx import TensorProto, helper, numpy_helper


def export_fasttext_lid(model_path, out_dir, int8=False):
    """Writes model.onnx and assets/{language.txt, vocab.json, config.json} for LanguageDetectorDL."""
    m = fasttext.load_model(model_path)
    a = m.f.getArgs()
    if str(a.loss) != "loss_name.softmax":
        raise ValueError(f"only softmax-loss fastText models can be exported, this one uses {a.loss}")
    if a.wordNgrams != 1:
        raise ValueError(f"only models trained with wordNgrams=1 can be exported, this one uses {a.wordNgrams}")

    os.makedirs(os.path.join(out_dir, "assets"), exist_ok=True)
    labels = [l.replace("__label__", "") for l in m.get_labels()]
    words = m.get_words(include_freq=False, on_unicode_error="surrogateescape")
    # a word that is not valid UTF-8 can never match real text
    vocab = [None if any(0xDC80 <= ord(c) <= 0xDCFF for c in w) else w for w in words]
    with open(os.path.join(out_dir, "assets", "language.txt"), "w", encoding="utf-8") as f:
        f.write("\n".join(labels) + "\n")
    with open(os.path.join(out_dir, "assets", "vocab.json"), "w", encoding="ascii") as f:
        json.dump(vocab, f, ensure_ascii=True)
    with open(os.path.join(out_dir, "assets", "config.json"), "w", encoding="ascii") as f:
        json.dump({"minn": a.minn, "maxn": a.maxn, "bucket": a.bucket, "nwords": len(words), "dim": a.dim}, f)

    # input_ids [batch, n] and mask [batch, n] -> mean of the input-matrix rows -> softmax probs [batch, labels]
    output_matrix = m.get_output_matrix().astype(np.float32)
    graph = helper.make_graph(
        [
            helper.make_node("Gather", ["input_matrix", "input_ids"], ["rows"]),
            helper.make_node("Unsqueeze", ["mask", "axis2"], ["mask3"]),
            helper.make_node("Mul", ["rows", "mask3"], ["masked"]),
            helper.make_node("ReduceSum", ["masked", "axis1"], ["summed"], keepdims=0),
            helper.make_node("ReduceSum", ["mask", "axis1"], ["count"], keepdims=1),
            helper.make_node("Div", ["summed", "count"], ["hidden"]),
            helper.make_node("MatMul", ["hidden", "output_matrix_t"], ["logits"]),
            helper.make_node("Softmax", ["logits"], ["probs"], axis=-1),
        ],
        "fasttext_lid",
        [
            helper.make_tensor_value_info("input_ids", TensorProto.INT64, ["batch", "features"]),
            helper.make_tensor_value_info("mask", TensorProto.FLOAT, ["batch", "features"]),
        ],
        [helper.make_tensor_value_info("probs", TensorProto.FLOAT, ["batch", len(labels)])],
        [
            numpy_helper.from_array(m.get_input_matrix().astype(np.float32), "input_matrix"),
            numpy_helper.from_array(np.ascontiguousarray(output_matrix.T), "output_matrix_t"),
            numpy_helper.from_array(np.array([1], dtype=np.int64), "axis1"),
            numpy_helper.from_array(np.array([2], dtype=np.int64), "axis2"),
        ],
    )
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)], ir_version=9)
    target = os.path.join(out_dir, "model.onnx")
    if not int8:
        onnx.save(model, target)
    else:
        from onnxruntime.quantization import QuantType, quantize_dynamic

        fp32 = os.path.join(out_dir, "model_fp32.onnx")
        onnx.save(model, fp32)
        # only the embedding table is quantized; the output layer stays fp32
        quantize_dynamic(fp32, target, op_types_to_quantize=["Gather"], weight_type=QuantType.QUInt8)
        os.remove(fp32)
    print(f"{out_dir}: {len(labels)} labels, {len(words)} words, model.onnx {os.path.getsize(target) / 1e6:.0f} MB")

In [ ]:
export_fasttext_lid(MODEL_PATH, EXPORT_DIR)

In [ ]:
!ls -lR {EXPORT_DIR}

## Import and save GlotLID in Spark NLP

- **Install and set up Spark NLP in Google Colab**

In [ ]:
!pip install -q pyspark==3.5.4 spark-nlp==7.0.1

In [ ]:
import sparknlp

spark = sparknlp.start()

print("Spark NLP version: {}".format(sparknlp.version()))
print("Apache Spark version: {}".format(spark.version))

- `loadSavedModel` reads the folder. It recognises the ONNX layout above; the CNN models of `LanguageDetectorDL` still load from TensorFlow exports.
- The settings below are the ones the pretrained `ld_glotlid_v4` model is saved with:
  - `setThreshold(0.7)` with `setThresholdLabel("unk")`: a result scored below 0.7 becomes `unk`
  - `setScriptPooling(True)`: for a non-Latin top label, the score is the sum of all labels in its script. GlotLID splits Chinese between Mandarin (`cmn_Hani`) and Cantonese (`yue_Hani`); pooling stops that split from pushing Chinese below the threshold.
  - `setMinLetters(20)`: when the top label is in Latin script (`_Latn`), text with fewer than 20 letters becomes `unk`. Very short fragments such as "No oral lesions." are often given a close language with a high score.
  - `setTopK(5)`: the metadata keeps the 5 best scores instead of all 2,060
  - `setCoalesceSentences(False)`: one result per sentence instead of one averaged result per document

In [ ]:
from sparknlp.annotator import LanguageDetectorDL

languageDetector = LanguageDetectorDL.loadSavedModel(EXPORT_DIR, spark) \
    .setInputCols(["document"]) \
    .setOutputCol("language") \
    .setThreshold(0.7) \
    .setThresholdLabel("unk") \
    .setScriptPooling(True) \
    .setMinLetters(20) \
    .setTopK(5) \
    .setCoalesceSentences(False)

- Let's save it on disk so it is easier to be moved around and also be used later via `.load` function

In [ ]:
languageDetector.write().overwrite().save("glotlid_v4_spark_nlp")

Let's clean up stuff we don't need anymore

In [ ]:
!rm -rf {EXPORT_DIR}

Awesome 😎 ! This is GlotLID v4 from HuggingFace 🤗 loaded and saved by Spark NLP 🚀

In [ ]:
!ls -l glotlid_v4_spark_nlp

Now let's see how we can use it on other machines, clusters, or any place you wish to use it 😊

In [ ]:
from pyspark.ml import Pipeline
from sparknlp.base import DocumentAssembler

languageDetector_loaded = LanguageDetectorDL.load("glotlid_v4_spark_nlp") \
    .setInputCols(["document"]) \
    .setOutputCol("language")

document_assembler = DocumentAssembler() \
    .setInputCol("text") \
    .setOutputCol("document")

pipeline = Pipeline(stages=[document_assembler, languageDetector_loaded])

texts = [
    "The patient was admitted with chest pain and shortness of breath for three days.",
    "患者为67岁男性,因胸痛和呼吸困难三天入院。",
    "El paciente ingresó por dolor torácico y dificultad para respirar desde hace tres días.",
    "No oral lesions.",
    "def mean(xs): return sum(xs) / len(xs)",
]
data = spark.createDataFrame([[t] for t in texts]).toDF("text")
result = pipeline.fit(data).transform(data)

result.selectExpr("text", "language.result[0] as language", "language.metadata[0]['confidence'] as confidence") \
    .show(truncate=60)

The scores match fastText's own predictions to within 0.0001. fastText ranks labels by `log(p + 1e-5)` and reports `exp` of that, so each probability it prints is 0.00001 higher than the plain softmax Spark NLP reports:

In [ ]:
import fasttext

ft = fasttext.load_model(MODEL_PATH)
for text, row in zip(texts, result.select("language").collect()):
    labels, probs = ft.predict(text, k=1)
    label = labels[0].replace("__label__", "")
    spark_score = float(row.language[0].metadata[label])
    print(f"{label:9} fastText {probs[0]:.6f}  Spark NLP {spark_score:.6f}  {text[:40]}")

## Optional: a smaller int8 model

Quantizing only the embedding table to 8 bits gives a 196 MB model. On our 452k-text parity set (FLORES-200 in all 204 languages, UDHR, medical text and edge cases) it kept fastText's top label for 99.8% of texts; the differences are on short, low-confidence fragments.

In [ ]:
export_fasttext_lid(MODEL_PATH, "glotlid_v4_int8_onnx", int8=True)

LanguageDetectorDL.loadSavedModel("glotlid_v4_int8_onnx", spark) \
    .setInputCols(["document"]) \
    .setOutputCol("language") \
    .setThreshold(0.7) \
    .setThresholdLabel("unk") \
    .setScriptPooling(True) \
    .setMinLetters(20) \
    .setTopK(5) \
    .setCoalesceSentences(False) \
    .write().overwrite().save("glotlid_v4_int8_spark_nlp")

!rm -rf glotlid_v4_int8_onnx

That's it! You can now use GlotLID in Spark NLP 🚀